# 02 — Preprocessing

Filter, ICA, and epoch the raw EEG runs into `(n_epochs, n_channels, n_times)` arrays saved to `data/X.npy` / `data/y.npy` for downstream feature extraction and model training.

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import numpy as np

from preprocessing import preprocess_run, epochs_to_arrays
from utils import LEG_IMAGERY_RUNS, get_logger, set_seed

set_seed(42)
logger = get_logger('preprocessing-nb')
DATA_DIR = Path('../data')

## Run the pipeline per subject/run

`preprocess_run` applies: bandpass (1-40Hz) → notch (50Hz) → ICA → epoching `[-0.5, 2.5s]` with baseline correction and peak-to-peak rejection.

In [ ]:
all_X, all_y = [], []

edf_files = sorted(DATA_DIR.glob('sub-*/**/*.edf'))
print(f'Found {len(edf_files)} EDF files')

for edf_path in edf_files:
    run_num = int(edf_path.stem.split('R')[-1][:2])
    if run_num not in LEG_IMAGERY_RUNS:
        continue
    try:
        epochs = preprocess_run(edf_path)
        X, y = epochs_to_arrays(epochs)
        all_X.append(X)
        all_y.append(y)
    except Exception as e:
        logger.warning(f'Skipping {edf_path.name}: {e}')

## Concatenate and save

In [ ]:
X = np.concatenate(all_X, axis=0)
y = np.concatenate(all_y, axis=0)
print('X shape:', X.shape)
print('y shape:', y.shape)
print('Class balance:', {int(c): int((y == c).sum()) for c in np.unique(y)})

np.save(DATA_DIR / 'X.npy', X)
np.save(DATA_DIR / 'y.npy', y)
print('Saved to data/X.npy and data/y.npy')